# 03 — Analyse d'expression différentielle (Alzheimer vs contrôle)

**Objectif :** identifier les gènes dont l'expression diffère significativement entre patients Alzheimer et contrôles, dans le cortex entorhinal.

On part des artefacts du notebook 02 (matrice log2 + quantile-normalisée au niveau sonde, detection calls, métadonnées). Deux décisions méthodologiques, **tranchées en amont**, sont appliquées ici :

1. **Filtrage ABS_CALL** : on ne garde que les sondes détectées (`Present`) dans au moins **k échantillons**, avec `k =` taille du plus petit groupe (calculée dynamiquement).
2. **Collapse sonde → gène** (stratégie *max mean*) : pour chaque gène, on garde la sonde de plus forte intensité moyenne. Les sondes **sans symbole** ou à **symboles multiples** sont écartées (jamais mappées au hasard).

Puis l'analyse différentielle proprement dite :
- **t-test de Welch** par gène (variances inégales, effectifs déséquilibrés),
- **log2 fold-change** = moyenne(AD) − moyenne(contrôle),
- **correction de Benjamini-Hochberg** (contrôle du FDR) pour les ~10 000 tests simultanés.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_loader, preprocessing as prep, stats as st

warnings.filterwarnings("ignore", message="Columns.*SPOT_ID.*mixed types")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Artefacts du notebook 02
expr = pd.read_parquet(PROCESSED_DIR / "ec_expr_norm_probe.parquet")
abscall = pd.read_parquet(PROCESSED_DIR / "ec_abscall_probe.parquet")
meta = pd.read_csv(PROCESSED_DIR / "ec_metadata.csv", index_col=0)

print(f"Matrice (niveau sonde) : {expr.shape[0]} sondes × {expr.shape[1]} échantillons")
print("Groupes :", meta["group"].value_counts().to_dict())

Matrice (niveau sonde) : 54675 sondes × 23 échantillons
Groupes : {'Control': 13, 'AD': 10}


## 1. Filtrage ABS_CALL

Le seuil `k` est calé sur la **taille du plus petit groupe** : ainsi une sonde qui ne serait détectée que chez les patients (ou que chez les contrôles) — donc potentiellement un marqueur de la maladie — n'est pas éliminée à tort.

In [2]:
group_counts = meta["group"].value_counts()
k = int(group_counts.min())  # seuil = taille du plus petit groupe
print(f"Effectifs : {group_counts.to_dict()}  ->  k = {k}")

expr_filt = prep.filter_by_abscall(expr, abscall, min_present=k)
print(f"Sondes avant filtrage : {expr.shape[0]}")
print(f"Sondes après filtrage (Present dans ≥ {k} échantillons) : {expr_filt.shape[0]} "
      f"({100*expr_filt.shape[0]/expr.shape[0]:.1f} %)")

Effectifs : {'Control': 13, 'AD': 10}  ->  k = 10
Sondes avant filtrage : 54675
Sondes après filtrage (Present dans ≥ 10 échantillons) : 19202 (35.1 %)


## 2. Collapse sonde → gène (max mean)

On récupère la correspondance sonde → symbole dans l'annotation de la plateforme GPL570, puis on applique la stratégie *max mean*. On quantifie explicitement combien de sondes sont écartées car **non annotées** ou à **symboles multiples**, pour garder la traçabilité.

In [3]:
# Annotation plateforme : ID de sonde -> symbole de gène
gse = data_loader.load_gse()
gpl = list(gse.gpls.values())[0]
probe_to_symbol = gpl.table.set_index("ID")["Gene Symbol"]

# Traçabilité : décompte des sondes écartées (parmi les sondes filtrées)
sym = probe_to_symbol.reindex(expr_filt.index).astype("string").str.strip()
n_no_symbol = int(sym.isna().sum() + (sym == "").sum())
n_multi = int(sym.str.contains("///", regex=False).fillna(False).sum())
print(f"Sondes filtrées : {expr_filt.shape[0]}")
print(f"  écartées — sans symbole de gène : {n_no_symbol}")
print(f"  écartées — symboles multiples (///) : {n_multi}")

genes = prep.collapse_probes_to_genes(expr_filt, probe_to_symbol)
print(f"\nMatrice finale : {genes.shape[0]} gènes uniques × {genes.shape[1]} échantillons")
print(f"Index de gènes unique : {genes.index.is_unique} | NaN : {int(genes.isna().sum().sum())}")

Sondes filtrées : 19202
  écartées — sans symbole de gène : 2070
  écartées — symboles multiples (///) : 1025

Matrice finale : 10571 gènes uniques × 23 échantillons
Index de gènes unique : True | NaN : 0


## 3. Analyse différentielle

Pour chaque gène : t-test de Welch (AD vs contrôle), log2 fold-change (moyenne AD − moyenne contrôle), puis correction de Benjamini-Hochberg. La fonction `stats.differential_expression` renvoie la table triée par p-value ajustée croissante.

In [4]:
ad_samples = [s for s in meta.index[meta["group"] == "AD"] if s in genes.columns]
ctrl_samples = [s for s in meta.index[meta["group"] == "Control"] if s in genes.columns]
print(f"AD : {len(ad_samples)} échantillons | Contrôles : {len(ctrl_samples)} échantillons")

results = st.differential_expression(genes, ad_samples, ctrl_samples,
                                     group_label="AD", ref_label="Control")
results.head()

AD : 10 échantillons | Contrôles : 13 échantillons


,mean_AD,mean_Control,log2FC,t_stat,p_value,p_adj
gene,,,,,,
LPHN3,8.566418,10.081490,-1.515072,-11.593115,1.907111e-10,0.000002
KIAA1211L,9.946111,12.059960,-2.113848,-10.630471,6.874640e-10,0.000003
TROVE2,12.093580,9.862095,2.231484,10.894991,1.722438e-09,0.000003
ADSL,8.289710,9.636442,-1.346732,-9.784664,2.893144e-09,0.000003
TMEM129,9.102444,8.012918,1.089526,9.932966,2.297408e-09,0.000003


In [5]:
# Bilan : combien de gènes significatifs ?
# Seuils : FDR < 5 % et |log2FC| >= 1 (soit un fold-change d'au moins 2x).
summary = st.summarize_hits(results, alpha=0.05, lfc_threshold=1.0)
print(summary.to_string())

print("\n--- Top 15 gènes les plus significatifs ---")
display(results.head(15).round(4))

print("--- Top 10 surexprimés chez AD (log2FC > 0, FDR<0.05) ---")
sig = results[results["p_adj"] < 0.05]
display(sig.sort_values("log2FC", ascending=False).head(10).round(4))

print("--- Top 10 sous-exprimés chez AD (log2FC < 0, FDR<0.05) ---")
display(sig.sort_values("log2FC").head(10).round(4))

genes_testes                10571
FDR<0.05                     4082
FDR<0.05 & |log2FC|>=1.0     2156
surexprimes_AD                990
sousexprimes_AD              1166

--- Top 15 gènes les plus significatifs ---


,mean_AD,mean_Control,log2FC,t_stat,p_value,p_adj
gene,,,,,,
LPHN3,8.5664,10.0815,-1.5151,-11.5931,0.0,0.0
KIAA1211L,9.9461,12.0600,-2.1138,-10.6305,0.0,0.0
TROVE2,12.0936,9.8621,2.2315,10.8950,0.0,0.0
ADSL,8.2897,9.6364,-1.3467,-9.7847,0.0,0.0
TMEM129,9.1024,8.0129,1.0895,9.9330,0.0,0.0
ALKBH6,8.3119,10.6392,-2.3273,-9.9758,0.0,0.0
PTP4A2,12.1536,10.5008,1.6528,10.6930,0.0,0.0
SDF4,8.0909,10.5919,-2.5010,-10.3966,0.0,0.0
MAPRE2,9.3873,11.2727,-1.8854,-9.7957,0.0,0.0


--- Top 10 surexprimés chez AD (log2FC > 0, FDR<0.05) ---


,mean_AD,mean_Control,log2FC,t_stat,p_value,p_adj
gene,,,,,,
ITPKB,9.9892,5.5766,4.4125,7.9556,0.0000,0.0000
FAM107B,11.6558,7.7840,3.8717,8.5962,0.0000,0.0001
SLCO1A2,8.5386,4.9359,3.6027,5.4589,0.0000,0.0005
C21orf91,10.3390,6.9136,3.4254,7.5802,0.0000,0.0000
ABCA1,10.7486,7.5687,3.1799,4.6918,0.0001,0.0013
ZIC1,10.0734,6.8935,3.1799,6.7772,0.0000,0.0001
APLNR,9.3880,6.3290,3.0589,5.9120,0.0000,0.0003
CD44,8.3365,5.4258,2.9107,4.9467,0.0001,0.0008
SPP1,11.3024,8.3939,2.9085,7.6186,0.0000,0.0000


--- Top 10 sous-exprimés chez AD (log2FC < 0, FDR<0.05) ---


,mean_AD,mean_Control,log2FC,t_stat,p_value,p_adj
gene,,,,,,
ACTL6B,4.7886,9.1913,-4.4027,-7.2588,0.0000,0.0000
CAMK1G,3.9580,8.3239,-4.3659,-7.6081,0.0000,0.0000
G6PD,4.4433,8.4328,-3.9896,-6.6770,0.0000,0.0001
YJEFN3,5.1469,9.1052,-3.9583,-8.0397,0.0000,0.0001
TBL3,4.4594,8.3342,-3.8749,-7.9492,0.0000,0.0000
FLJ22184,6.5854,10.3734,-3.7879,-7.2984,0.0000,0.0000
SMYD5,4.5451,8.2979,-3.7528,-8.7891,0.0000,0.0000
MAST1,4.4766,8.1740,-3.6974,-6.7961,0.0000,0.0002
CHRM1,6.8863,10.4678,-3.5815,-7.6698,0.0000,0.0000


## 4. Sauvegarde des résultats

On enregistre :
- la **table complète** des résultats différentiels (`results/tables/`, versionnée — c'est un livrable du projet) ;
- la **matrice gènes × échantillons** finale (`data/processed/`, non versionnée), réutilisée par le notebook 04 pour les visualisations (volcano, heatmap, PCA).

In [6]:
results.to_csv(TABLES_DIR / "de_results_entorhinal.csv")
genes.to_parquet(PROCESSED_DIR / "ec_expr_gene_level.parquet")

print("Enregistré :")
print(f"  results/tables/de_results_entorhinal.csv  ({len(results)} gènes)")
print(f"  data/processed/ec_expr_gene_level.parquet  ({genes.shape[0]} × {genes.shape[1]})")

Enregistré :
  results/tables/de_results_entorhinal.csv  (10571 gènes)
  data/processed/ec_expr_gene_level.parquet  (10571 × 23)


## Bilan

À partir de la matrice normalisée au niveau sonde, le filtrage ABS_CALL puis le collapse *max mean* ont produit une matrice propre **gènes × échantillons**. Le t-test de Welch corrigé par Benjamini-Hochberg fournit, pour chaque gène, son log2 fold-change et son FDR.

La table de résultats est le socle du **notebook 04** : volcano plot (vue d'ensemble des gènes différentiels), heatmap des top gènes, PCA des échantillons, et confrontation à la signature Alzheimer connue de la littérature.